# Papyrus-Layoytstudien

In [1]:
from pathlib import Path
import sys
import xml.etree.ElementTree as ET
import json

# Das Notebook liegt im Unterordner scripts/. Verzeichnis robust bestimmen,
# egal ob der Kernel aus scripts/ oder aus der Projektwurzel gestartet wird.
SCRIPT_DIR = Path.cwd()
if SCRIPT_DIR.name != 'scripts' and (SCRIPT_DIR / 'scripts').is_dir():
    SCRIPT_DIR = SCRIPT_DIR / 'scripts'
sys.path.insert(0, str(SCRIPT_DIR))   # Module (page_xml_to_json etc.) importierbar machen

PROJECT_ROOT = SCRIPT_DIR.parent       # Projektwurzel, eine Ebene ueber scripts/
PAGE_XML_DIR = PROJECT_ROOT / 'page_xml'
DATA_FILE = PROJECT_ROOT / 'data' / 'layout_data.json'
DATA_FILE.parent.mkdir(parents=True, exist_ok=True)  # data/ anlegen, falls noch nicht vorhanden

ns = {'p': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

# re_page_xml = re.compile(r"^P.*\.xml$")

## Pipeline & Ausführungsreihenfolge

Die Zellen bilden eine **Pipeline**, die `layout_data.json` schrittweise aufbaut. Jede Verarbeitungsfunktion **liest die JSON selbst ein und schreibt sie zurück** – es wird nichts mehr über eine `layout_data`-Variable weitergereicht. Führe die Zellen **von oben nach unten** aus; lässt du eine frühere Zelle erneut laufen, führe **alle darunterliegenden erneut aus**.

Reihenfolge und Abhängigkeiten:

1. **PAGE-XML → JSON** (`collect_layout_data`) – Grundlage; erzeugt/aktualisiert `column_data` (Polygone, Zeilen), Bildmaße, px/cm. **Muss zuerst laufen.**
2. **Image Processing** (`process_layout_data`) – braucht `image_file`; schreibt Bbox-, Flächen- und Randmaße (Top-Level je Platte).
3. **Kolumnen-Merkmale** (`measure_columns`) – **braucht die Bbox aus Schritt 2**; schreibt Merkmale *in die einzelnen Kolumnen* (`column['metrics']`).
4. **Kolumnenneigung** (`measure_tilt`) – braucht nur `column_data`/Polygone/Zeilen aus Schritt 1 (unabhängig von 2–3); schreibt `column['tilt']`.
5. **Überblick** (`generate_overview`) – liest alles; **zuletzt** ausführen.

> ⚠ **Wichtig:** Ein erneuter Lauf von Schritt 1 (`collect_layout_data`) **ersetzt `column_data` komplett** und löscht damit die *pro Kolumne* berechneten Felder (`tilt`, `metrics`). Top-Level-Felder (Bbox, `tilt_reference`) bleiben erhalten. Nach jedem erneuten Schritt 1 daher die Schritte 3 und 4 (und 5) erneut ausführen – die Funktion warnt, wenn sie dabei berechnete Felder verwirft.

# PAGE XML to JSON

Die Auswertung der PAGE-XML-Dateien ist in das Modul `page_xml_to_json.py` ausgelagert. Es erfasst pro Platte Bildmaße, px/cm-Skala und die Textregionen – getrennt nach Kolumnen (`column_data`) und Fragmenten (`fragment_data`) – und schreibt das Ergebnis nach `DATA_FILE`.

Die Liste der auszuwertenden Dateien (`page_xml_files`) wird oben im Notebook definiert und als Argument an `collect_layout_data(...)` übergeben. Bestehende Einträge in der JSON bleiben erhalten (z. B. manuelle bbox-Felder und Messwerte); neue Platten kommen automatisch hinzu.

In [2]:
# find page xml in `page` subfolders of `page_xml_dir`
page_xml_files = list(PAGE_XML_DIR.rglob('page/*.xml'))
for file in page_xml_files:
    print(file)

c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0001_P_09782-Pl-A_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0002_P_09782-Pl-B_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0003_P_09782-Pl-C_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0004_P_09782-Pl-D_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0005_P_09782-Pl-E_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0006_P_09782-Pl-F_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0007_P_09782-Pl-G1_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0008_P_09782-Pl-H_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0009_P_09782-Pl-N2_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0010_P_09782-Pl-

In [3]:
import page_xml_to_json as pxj

# `page_xml_files` wird oben definiert und hier als Argument uebergeben.
layout_data = pxj.collect_layout_data(page_xml_files, data_file=DATA_FILE)

Processing page: 0001_P_09782-Pl-A_R_001
Processing page: 0002_P_09782-Pl-B_R_001
Processing page: 0003_P_09782-Pl-C_R_001
Processing page: 0004_P_09782-Pl-D_R_001
Processing page: 0005_P_09782-Pl-E_R_001
Processing page: 0006_P_09782-Pl-F_R_001
Processing page: 0007_P_09782-Pl-G1_R_001
Processing page: 0008_P_09782-Pl-H_R_001
Processing page: 0009_P_09782-Pl-N2_R_001
Processing page: 0010_P_09782-Pl-O_R_001
Processing page: 0011_P_09782-Pl-P_R_001
Processing page: 0012_P_09782-Pl-Q_R_001
Processing page: 0013_P_09782-Pl-R_R_001
Processing page: 0014_P_09782-Pl-S_R_001
Processing page: 0015_P_09782-Pl-T_R_001
Writing 15 pages (62 columns, 4 fragments) to c:\Users\Fabi\Repos\papyrus-layout-studien\layout_data.json


# Image Processing

Die Bildverarbeitung ist in das Modul `image_processing.py` ausgelagert. Es segmentiert das Fragment vom hellen Hintergrund, erzeugt eine Binärmaske und leitet daraus Bounding-Box, Flächen- und Randmaße ab. Der Aufruf erfolgt weiter unten über `process_layout_data(...)`.

## Einstellbare Parameter

Alle Stellschrauben sind in der Dataclass `SegmentationParams` gebündelt und können beim Aufruf überschrieben werden, z. B. `ip.SegmentationParams(min_component_ratio=0.05)`.

- **`strip_threshold_factor`** (Default `0.85`): Erkennung des Info-Streifens am unteren Rand. Wird der Streifen nicht sauber entfernt, Wert senken.
- **`gray_close_kernel_size`** (Default `(51, 51)`): Überbrückt helle Stellen im Fragment. Wird das Fragment von hellen Flecken „aufgefressen“, vergrößern.
- **`binary_fill_kernel_size`** (Default `(15, 15)`): Füllt kleine Löcher und Risse in der Maske.
- **`min_component_ratio`** (Default `0.02`): Rauschfilter. Bei vielen kleinen Artefakten (wie in `P_09782-Pl-H_R_001`) erhöhen (z. B. `0.05` oder `0.1`), um nur die größten Teile zu behalten.

Mit `overwrite_masks=True` werden zwischengespeicherte Masken neu berechnet; ansonsten werden vorhandene Masken aus `images/masks/` wiederverwendet.

## Manual Bounding Box Mode

For fragments where automatic segmentation is unreliable, you can manually define the bounding box in `layout_data.json`. The processor will then use this data instead of running the image segmentation pipeline.

### How to Use

Add these fields to the relevant entry in `layout_data.json`:

```json
{
  "image_key": {
    "bbox_measurement": "manual",
    "bbox_manual": {
      "origin": [x, y],
      "width": w,
      "height": h
    },
    ...rest of data...
  }
}
```

**Fields:**
- `bbox_measurement`: Set to `"manual"` to enable manual mode; set to `"automatic"` or omit for automatic segmentation
- `bbox_manual.origin`: `[x, y]` - top-left corner of the bounding box
- `bbox_manual.width`: `w` - width in pixels
- `bbox_manual.height`: `h` - height in pixels

### Behavior

When `bbox_measurement` is set to `"manual"`:
- The image segmentation pipeline is **skipped**
- No binary mask is created or saved
- All derived measurements (margins, coverage) are computed from the manual bbox
- `coverage_pct` is set to 100% (assuming the entire bbox is fragment)
- The preview image is still generated with the bbox overlaid

### Example

```json
"0008_P_09782-Pl-H_R_001": {
  "bbox_measurement": "manual",
  "bbox_manual": {
    "origin": [120, 80],
    "width": 1200,
    "height": 1500
  },
  ...
}
```

In [ ]:
import image_processing as ip

# Segmentierungsparameter (bei Bedarf anpassen, siehe Dokumentation oben).
params = ip.SegmentationParams()

# Dateibasiert: liest DATA_FILE, verarbeitet alle Eintraege, legt Masken/
# Vorschaubilder unter images/masks/ ab und schreibt die Masse zurueck nach DATA_FILE.
layout_data = ip.process_layout_data(
    data_file=DATA_FILE,
    project_root=PROJECT_ROOT,
    params=params,
    overwrite_masks=False,  # True regeneriert alle Masken
)

# Kolumnen-Merkmale

Einfache Layout-Merkmale je Kolumne und Kolumnenpaar, ausgelagert in `column_metrics.py`. Läuft **nach** der Bounding-Box-Berechnung, da oberer/unterer Rand und die Schriftspiegel-Verhältnisse das Blattmaß (Bbox) brauchen.

Pro Kolumne: Höhe, Breite, Fläche, oberer/unterer Rand, Schriftspiegel-Verhältnis (Höhe & Fläche). Pro Nachbarpaar (in `to_next` der linken Kolumne): Intercolumn-Abstand, Kolumne-zu-Kolumne-Breite und -Fläche. Werte in px und – wo eine Skala vorliegt – cm/cm²; Ränder und Schriftspiegel werden bei Mehrfach-Fragment-Platten (z. B. T) unterdrückt.

Horizontale Abstände werden in der Mitte des y-Überlappungsbereichs benachbarter Kanten gemessen (= Mittel über die Überlappung, da die Kanten gerade sind).

In [ ]:
import column_metrics as cm

# Dateibasiert: liest DATA_FILE (nach der Bbox-Berechnung), berechnet einfache
# Merkmale je Kolumne/Paar und schreibt sie zurueck nach DATA_FILE.
layout_data = cm.measure_columns(data_file=DATA_FILE)

# Kolumnenneigung (Maas's Law)

Neigungswinkel der linken Kante jeder **nutzbaren** Kolumne, ausgelagert in `column_tilt.py`. Zwei Varianten (vgl. Methodik-Notiz „Winkelmessung Maas's Law"):

1. `tilt_vs_vertical_deg` – gegen die reine Bild-Senkrechte (Annahme: das Digitalisat ist lotrecht ausgerichtet).
2. `tilt_vs_ideal_deg` – gegen die Senkrechte einer plattenweiten Ideal-Horizontalen, gemittelt aus den **Oberkanten** der nutzbaren Kolumnen (entfernt eine globale Plattenschiefe).

Ergebnis je Kolumne in `column['tilt']`, je Platte in `entry['tilt_reference']` (u. a. `plate_skew_deg` = geschätzte Plattenschiefe = Differenz beider Varianten). Vorzeichen: **positiv = untere Kante nach links versetzt**. Erwartung: ~1–4° pro Kolumne.

In [5]:
import column_tilt as ct

# Dateibasiert: liest DATA_FILE, berechnet die Kolumnenneigung (Maas's Law) je
# nutzbarer Kolumne und schreibt sie zurueck nach DATA_FILE.
layout_data = ct.measure_tilt(data_file=DATA_FILE)

[0001_P_09782-Pl-A_R_001] 5 nutzbare Kolumnen, Plattenschiefe 0.842° | Baseline-Horizontale 1.136° (Δ Oberkante−Baseline -0.294°, 5 Kol. mit Zeilen)
[0002_P_09782-Pl-B_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.909° | Baseline-Horizontale 1.241° (Δ Oberkante−Baseline -0.332°, 4 Kol. mit Zeilen)
[0003_P_09782-Pl-C_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.286°
[0004_P_09782-Pl-D_R_001] 4 nutzbare Kolumnen, Plattenschiefe -0.045°
[0005_P_09782-Pl-E_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.203°
[0006_P_09782-Pl-F_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.238°
[0007_P_09782-Pl-G1_R_001] 2 nutzbare Kolumnen, Plattenschiefe 0.603°
[0008_P_09782-Pl-H_R_001] 11 nutzbare Kolumnen, Plattenschiefe -0.165°
[0009_P_09782-Pl-N2_R_001] 2 nutzbare Kolumnen, Plattenschiefe -0.464°
[0010_P_09782-Pl-O_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.442°
[0011_P_09782-Pl-P_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.39°
[0012_P_09782-Pl-Q_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.79°
[0013_P_

# Datenüberblick (Markdown + CSV-Export)

Aus `layout_data.json` werden zwei Tabellen erzeugt (ausgelagert in `layout_overview.py`):

1. **Zusammenfassung pro Platte** – eine Zeile je Platte (Kolumnen-/Fragmentzahl, Höhe/Breite, Zeilen-Kennzahlen, mittlere Neigung der drei Ansätze, Platten- und Baseline-Schiefe). Gespeichert als `layout_overview.md` **und** `layout_summary_per_plate.csv`.
2. **Daten pro Kolumne** – eine Zeile je Kolumne (Neigung aller drei Ansätze plus die Kolumnen-Merkmale aus `measure_columns`). Gespeichert als `layout_per_column.csv`.

`generate_overview(...)` lädt die Daten, baut beide Tabellen, schreibt Markdown + CSVs und gibt den Markdown-String zurück. `IPython.display` kann CSV nicht direkt rendern; die Zelle zeigt daher die Zusammenfassung als Markdown und die beiden CSVs als `FileLink` (zum Öffnen/Herunterladen). Als gerenderte Tabelle liessen sie sich mit pandas anzeigen (`display(pd.read_csv(...))`).

In [2]:
import layout_overview as lo
from IPython.display import Markdown, display, FileLink

# Erzeugt die Zusammenfassung pro Platte als Markdown (layout_overview.md) und
# zusaetzlich zwei CSV-Tabellen neben der JSON:
#   layout_summary_per_plate.csv  (eine Zeile pro Platte)
#   layout_per_column.csv         (eine Zeile pro Kolumne)
markdown = lo.generate_overview(
    data_file=DATA_FILE,
    overview_file=PROJECT_ROOT / 'layout_overview.md',
)
display(Markdown(markdown))

# CSV kann IPython nicht direkt rendern -> Links zum Öffnen/Herunterladen.
# (Als Tabelle anzeigen ginge mit pandas: display(pd.read_csv(pfad)).)
display(FileLink(str(PROJECT_ROOT / 'layout_summary_per_plate.csv')))
display(FileLink(str(PROJECT_ROOT / 'layout_per_column.csv')))

Ueberblick (Markdown) -> c:\Users\Fabi\Repos\papyrus-layout-studien\layout_overview.md
Zusammenfassung pro Platte (CSV) -> c:\Users\Fabi\Repos\papyrus-layout-studien\layout_summary_per_plate.csv
Daten pro Kolumne (CSV) -> c:\Users\Fabi\Repos\papyrus-layout-studien\layout_per_column.csv


# Layout-Überblick – 2026-07-07

Datenquelle: `layout_data.json`

## Gesamtüberblick

- Platten: 15
- Kolumnen gesamt: 62 (davon nutzbar: 55)
- Fragmente gesamt: 4
- Kolumnen mit erfassten Zeilen: 10 (ohne Zeilendaten: 52)
- Zeilen gesamt: 457
- Ø Zeilen pro Kolumne: 45.7
- Zeilen pro Kolumne (min / max): 25 / 53

> Ø, min und max beziehen sich nur auf Kolumnen mit erfassten Zeilen. Fragmente (z. B. Platte T) werden separat gezaehlt und fliessen nicht in die Kolumnen-Statistik ein. Höhe/Breite (cm) sind aus den Bbox-Pixelmaßen und der px/cm-Skala abgeleitet (— ohne Skala oder bei mehreren Fragmenten). Ø Neigung = mittlere Kolumnenneigung der nutzbaren Kolumnen gegen drei Referenzen: die Bild-Senkrechte (vert.), die plattenweite Ideal-Horizontale aus den Oberkanten (ideal) und die Schrifthorizontale der jeweiligen Kolumne aus ihren Baselines (baseline, Johnson-Methode; nur wo Zeilen erfasst sind, sonst —). Schiefe = geschätzte Plattenschiefe aus den Oberkanten; Baseline-Schiefe = plattenweite Schrifthorizontale aus den Baselines (Kontrolle zur Oberkanten-Schiefe, in Klammern die Zahl der Kolumnen mit Zeilen). Neigung positiv = untere Kante nach links. Teile = Zahl zusammenhängender Maskenteile (Fragmentierungsmaß: 1 = zusammenhängendes Blatt, >1 = physisch getrennte Stücke; — im manuellen Bbox-Modus). Auf stark fragmentierten Platten (viele unbenutzbare Kolumnen, Teile > 1) sind die Neigungswinkel nur mit geringer Konfidenz zu lesen.

## Pro Platte

| Platte | Kolumnen | nutzbar | Fragmente | Teile | Höhe (cm) | Breite (cm) | Zeilen gesamt | Ø Zeilen/Kol | min | max | Ø Neig. vert. (°) | Ø Neig. ideal (°) | Ø Neig. baseline (°) | Schiefe (°) | Baseline-Schiefe (°) |
|--------|---------:|--------:|----------:|------:|----------:|------------:|--------------:|-------------:|----:|----:|------------------:|------------------:|---------------------:|------------:|---------------------:|
| A | 6 | 5 | 0 | 1 | 29.1 | 44.5 | 278 | 46.3 | 25 | 53 | 2.7 | 1.9 | 1.6 | 0.84 | 1.14 (5) |
| B | 4 | 4 | 0 | 1 | 29.2 | 34.0 | 179 | 44.8 | 44 | 46 | 3.1 | 2.2 | 1.9 | 0.91 | 1.24 (4) |
| C | 4 | 4 | 0 | 1 | 29.9 | 34.6 | 0 | — | — | — | 4.0 | 3.7 | — | 0.29 | — |
| D | 4 | 4 | 0 | 1 | 30.1 | 35.0 | 0 | — | — | — | 3.1 | 3.1 | — | -0.04 | — |
| E | 4 | 4 | 0 | 1 | 30.5 | 34.0 | 0 | — | — | — | 3.0 | 2.8 | — | 0.20 | — |
| F | 4 | 4 | 0 | 1 | 30.4 | 34.5 | 0 | — | — | — | 2.1 | 1.9 | — | 0.24 | — |
| G1 | 3 | 2 | 0 | 1 | 30.4 | 25.6 | 0 | — | — | — | 2.2 | 1.6 | — | 0.60 | — |
| H | 11 | 11 | 0 | — | — | — | 0 | — | — | — | 2.3 | 2.5 | — | -0.17 | — |
| N2 | 2 | 2 | 0 | 2 | 31.0 | 17.2 | 0 | — | — | — | 2.6 | 3.0 | — | -0.46 | — |
| O | 4 | 4 | 0 | 1 | 31.5 | 34.1 | 0 | — | — | — | 2.2 | 1.7 | — | 0.44 | — |
| P | 4 | 4 | 0 | 1 | 30.7 | 33.7 | 0 | — | — | — | 1.6 | 1.2 | — | 0.39 | — |
| Q | 4 | 4 | 0 | 1 | 31.2 | 34.3 | 0 | — | — | — | 2.1 | 1.4 | — | 0.79 | — |
| R | 5 | 2 | 0 | 3 | 26.8 | 33.7 | 0 | — | — | — | 0.7 | 1.4 | — | -0.73 | — |
| S | 3 | 1 | 0 | 3 | 30.6 | 20.7 | 0 | — | — | — | 0.7 | 1.1 | — | -0.36 | — |
| T | 0 | 0 | 4 | 4 | — | — | 0 | — | — | — | — | — | — | — | — |


c:\Users\Fabi\Repos\papyrus-layout-studien\layout_summary_per_plate.csv

c:\Users\Fabi\Repos\papyrus-layout-studien\layout_per_column.csv